# Old English vowel length: ByT5 context pilot (Colab)

Five context arms (sentence, word, left, two words each side, five words each side), each trained for one epoch on the
same fixed subset of 10,000 examples. All logic is in `src/oevl/pilot.py`; this notebook only puts the project on the
machine and calls it.

1. Runtime > Change runtime type > **T4 GPU**.
2. Set `MAX_BYTES` below, then Run all, and choose a zip of the repository folder when asked.
3. To resume after a session ended, first upload the earlier `encoder_pilot_<bytes>_results.zip` to `/content` with
   the file browser on the left.

This is an exploratory pilot, not a full-data replication. There is no checkpoint inside an arm.

In [ ]:
MAX_BYTES = 128  # 128 or 256: the only setting that differs between the two pilots

import shutil, subprocess, sys, zipfile
from pathlib import Path
from google.colab import files

MARKER = 'src/oevl/pilot.py'
root = Path('/content/oe_project')
if not (root / MARKER).is_file():
    print('Choose a zip of the repository folder.')
    zips = [Path('/content') / name for name in files.upload() if name.endswith('.zip')]
    zips = [z for z in zips if any(n.endswith(MARKER) for n in zipfile.ZipFile(z).namelist())]
    assert len(zips) == 1, f'expected one project zip, got {zips}'
    with zipfile.ZipFile(zips[0]) as z:
        prefix = next(n for n in z.namelist() if n.endswith(MARKER))[:-len(MARKER)]
        z.extractall('/content/_project_zip')
    shutil.copytree(Path('/content/_project_zip') / prefix, root, dirs_exist_ok=True)
assert (root / MARKER).is_file(), 'project not found'
print('PROJECT READY:', root)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(root / 'requirements-neural.txt')], check=True)

In [ ]:
import os
out = Path(f'/content/encoder_pilot_{MAX_BYTES}_results')
earlier = list(Path('/content').glob(out.name + '.zip'))
cmd = [sys.executable, '-m', 'oevl.pilot', '--max-bytes', str(MAX_BYTES), '--out', str(out)]
cmd += ['--restore', str(earlier[0])] if len(earlier) == 1 else []
env = dict(os.environ, PYTHONPATH=str(root / 'src'))
p = subprocess.Popen(cmd, cwd=root, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    print(line, end='', flush=True)
if p.wait():
    raise RuntimeError(f'pilot stopped ({p.returncode}); finished arms are in {out}.zip')

## Download

Run this after any finished arm. A reset of the runtime deletes the archive.

In [ ]:
zip_path = Path(f'/content/encoder_pilot_{MAX_BYTES}_results.zip')
if zip_path.is_file():
    files.download(str(zip_path))
else:
    print('no finished arm yet:', zip_path)